# Práctica — Clase 6: cambia la "personalidad" de tu chatbot

## Contexto

El `lab-instructions.md` real del material fuente (el mismo que usaste para
construir el chatbot de SmolLM2 en la clase) sugiere textualmente
experimentar con el **system prompt**, la **temperature** y el **top_p**
para cambiar cómo se comporta el bot — por ejemplo, haciéndolo "más amigable"
cambiando el mensaje de sistema y subiendo `temperature` a 0.9 y `top_p` a
0.95.

En esta práctica vas a hacer exactamente eso: crear al menos una
"personalidad" nueva para el chatbot de `HuggingFaceTB/SmolLM2-360M-Instruct`
y comparar sus respuestas contra las del bot con la configuración por
defecto, usando las mismas preguntas de prueba.

## Paso 0: instalar dependencias y cargar el modelo (igual que en la clase)

In [ ]:
!pip install transformers==4.41.2 torch==2.2.2 accelerate==0.30.1 numpy==1.26.4

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import warnings

warnings.filterwarnings("ignore")

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.unk_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Modelo cargado.")

## Andamiaje: función de una sola respuesta

Para comparar "personalidades" rápidamente, esta función arma una
conversación de un solo turno (system + user) con los parámetros que le
pases, y devuelve la respuesta del bot. Así no necesitas un bucle
interactivo `while True` para probar varias configuraciones seguidas.

In [ ]:
def generar_respuesta(mensaje_usuario, system_prompt, temperature=0.5, top_p=0.8, max_new_tokens=60):
    """Genera una respuesta de SmolLM2 para un único turno de conversación,
    dado un system_prompt y parámetros de muestreo (temperature, top_p)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": mensaje_usuario}
    ]

    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        max_length=512
    )

    with torch.inference_mode():
        outputs = model.generate(
            tokenized["input_ids"],
            attention_mask=tokenized["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(
        outputs[0][tokenized["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )


# Preguntas de prueba: las usaremos con distintas personalidades
preguntas_de_prueba = [
    "What should I have for dinner tonight?",
    "Tell me about your day."
]

## Personalidad por defecto (línea base)

Esta es la configuración que usaste en la clase: un asistente útil y
conciso, con `temperature=0.5` y `top_p=0.8`. Ejecuta esta celda para tener
un punto de comparación.

In [ ]:
system_prompt_base = "You are a helpful AI assistant. Give short and concise answers in 2-3 lines."

print("=== Personalidad BASE ===\n")
for pregunta in preguntas_de_prueba:
    respuesta = generar_respuesta(pregunta, system_prompt_base, temperature=0.5, top_p=0.8)
    print(f"> {pregunta}\nBot: {respuesta}\n")

## Ejercicio: crea tu propia personalidad

**Enunciado**: define un nuevo `system_prompt` que le dé al bot una
personalidad claramente distinta a la base (por ejemplo: pirata, mayordomo
extremadamente formal, entrenador deportivo hiperentusiasta, poeta
melancólico...). Elige también tus propios valores de `temperature` y
`top_p` — recuerda que valores más altos (p. ej. 0.9 / 0.95, como sugiere el
material fuente) producen respuestas más variadas y creativas, mientras que
valores más bajos las hacen más predecibles.

Genera respuestas para las mismas `preguntas_de_prueba` con tu configuración
y compáralas con la personalidad base de arriba.

Pistas:
- Reutiliza `generar_respuesta(mensaje, system_prompt, temperature, top_p)`.
- No hace falta recargar el modelo: ya está cargado en memoria.

In [ ]:
# TODO: escribe un system prompt que le dé al bot una personalidad distinta
mi_system_prompt = None  # TODO

# TODO: elige tus propios valores de temperature y top_p
mi_temperature = None  # TODO
mi_top_p = None  # TODO

# TODO: genera y muestra las respuestas para preguntas_de_prueba
# usando tu configuración, igual que se hizo con la personalidad base

<details>
<summary>💡 Ver solución</summary>

```python
# Ejemplo de personalidad: un pirata muy expresivo, con respuestas más
# creativas (temperature y top_p más altos, tal como sugiere el
# lab-instructions.md real del material fuente)
mi_system_prompt = (
    "You are a cheerful pirate captain. Answer every question in pirate "
    "speak, full of enthusiasm, in 2-3 lines."
)
mi_temperature = 0.9
mi_top_p = 0.95

print("=== Mi personalidad ===\n")
for pregunta in preguntas_de_prueba:
    respuesta = generar_respuesta(
        pregunta,
        mi_system_prompt,
        temperature=mi_temperature,
        top_p=mi_top_p
    )
    print(f"> {pregunta}\nBot: {respuesta}\n")
```

Al comparar contra la personalidad base deberías notar dos efectos
distintos y superpuestos:

1. **El tono cambia** por el `system_prompt` (formal/conciso vs. pirata
   entusiasta) — esto es casi enteramente responsabilidad del prompt.
2. **La variedad de las respuestas cambia** por `temperature` y `top_p` — con
   valores más altos, vuelve a ejecutar la misma pregunta varias veces y
   observa cómo las respuestas cambian más de una ejecución a otra.

Con un modelo tan pequeño (360M parámetros) no esperes una actuación
perfecta del personaje en todas las respuestas — es normal que a veces
"se salga del personaje". Es una limitación real del modelo, no un error de
tu código.
</details>

## Para seguir

- Vuelve al notebook de clase:
  [clase06_llms_chatbot_huggingface.ipynb](../notebooks_clase/clase06_llms_chatbot_huggingface.ipynb).
- Repasa la teoría en
  [manual/09_llms_gpt_llama.md](../manual/09_llms_gpt_llama.md) y
  [manual/10_chatbot_huggingface.md](../manual/10_chatbot_huggingface.md).
- En la próxima clase vas a exponer un chatbot como este a través de un
  servidor web con Flask, para que puedas hablar con él desde una página
  HTML en vez de la terminal.